# SQL Business Analysis

This notebook uses the SQLite database created during schema exploration to answer business questions about orders, delivery performance, customer satisfaction, seller performance, and product-category behavior.

The main focus is not only writing SQL queries, but also being careful about row grain. Some queries are order-level, some become item-level after joining `order_items`, and some become item-review-level after joining `reviews`.

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/processed/olist.db")

conn = sqlite3.connect(DB_PATH)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

## Database Connection Check

Before starting the analysis, the notebook checks that the SQLite database is accessible and that the expected tables are available.

In [2]:
pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

,name
0,category_translation
1,customers
2,geolocation
3,order_items
4,orders
5,payments
6,products
7,reviews
8,sellers


## Order Status Overview

The first analysis checks how orders are distributed across different statuses. This helps separate completed orders from incomplete, canceled, or still-in-progress orders.

In [3]:
order_status_query = """
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 2) AS order_percentage
FROM orders
GROUP BY order_status
ORDER BY total_orders DESC;
"""

order_status_df = pd.read_sql_query(order_status_query, conn)

order_status_df

,order_status,total_orders,order_percentage
0,delivered,96478,97.02
1,shipped,1107,1.11
2,canceled,625,0.63
3,unavailable,609,0.61
4,invoiced,314,0.32
5,processing,301,0.30
6,created,5,0.01
7,approved,2,0.00


### Order Status Observation

Most orders in the dataset are marked as delivered, meaning the dataset is heavily dominated by completed orders. Non-delivered statuses such as shipped, canceled, unavailable, invoiced, processing, created, and approved represent a small minority of total orders.

For future delivery-time and revenue analysis, delivered orders should often be analyzed separately from incomplete or canceled orders.

Small Note: Reason why I keep spelling cancelled as 'canceled' is because this is the variable name used in the database.

## Delivered vs Not Delivered Orders

The next analysis simplifies order status into two groups: delivered orders and not delivered orders. This helps estimate how much of the dataset is usable for completed-order analysis.

In [4]:
delivered_status_query = """
WITH categorized_orders AS (
    SELECT 
        order_id,
        CASE 
            WHEN order_status = 'delivered' THEN 'Delivered'
            ELSE 'Not Delivered'
        END AS delivery_group
    FROM orders
)
SELECT 
    delivery_group,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS order_percentage
FROM categorized_orders
GROUP BY delivery_group
ORDER BY total_orders DESC;
"""

delivered_status_df = pd.read_sql_query(delivered_status_query, conn)

delivered_status_df

,delivery_group,total_orders,order_percentage
0,Delivered,96478,97.02
1,Not Delivered,2963,2.98


### Delivered vs Not Delivered Observation

Most orders in the dataset are delivered, while only a small percentage are not delivered. This confirms that the dataset is mainly composed of completed orders.

For future analyses involving delivery time, delays, revenue, and customer reviews, delivered orders should usually be analyzed separately from non-delivered orders to avoid mixing completed and incomplete order lifecycles.

## Monthly Order Trend

In [5]:
monthly_orders_query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS purchase_month,
    COUNT(*) AS total_orders
FROM orders AS o
GROUP BY purchase_month
ORDER BY purchase_month;
"""

monthly_orders_df = pd.read_sql_query(monthly_orders_query, conn)

monthly_orders_df

,purchase_month,total_orders
0,2016-09,4
1,2016-10,324
2,2016-12,1
3,2017-01,800
4,2017-02,1780
5,2017-03,2682
6,2017-04,2404
7,2017-05,3700
8,2017-06,3245
9,2017-07,4026


### Monthly Order Trend Observation

Order volume increased strongly from early 2017 and remained high throughout most of 2018. November 2017 appears to be one of the strongest months in the dataset.

The earliest and latest months contain very few orders, which likely means they are partial months rather than true business declines. Because of this, trend analysis should focus mainly on the period from 2017-01 to 2018-08.

In [6]:
monthly_delivered_orders_query = """
SELECT
    strftime('%Y-%m', o.order_purchase_timestamp) AS purchase_month,
    COUNT(*) AS total_delivered_orders
FROM orders AS o
WHERE o.order_status = 'delivered'
GROUP BY purchase_month
ORDER BY purchase_month;
"""

monthly_delivered_orders_df = pd.read_sql_query(
    monthly_delivered_orders_query,
    conn
)

monthly_delivered_orders_df

,purchase_month,total_delivered_orders
0,2016-09,1
1,2016-10,265
2,2016-12,1
3,2017-01,750
4,2017-02,1653
5,2017-03,2546
6,2017-04,2303
7,2017-05,3546
8,2017-06,3135
9,2017-07,3872


### Monthly Delivered Order Trend Observation

Delivered order volume follows a similar pattern to total order volume, increasing strongly through 2017 and remaining high across most of 2018.

The delivered-order trend ends at 2018-08, while the total-order trend included a very small number of orders in 2018-09 and 2018-10. This means that the final months in the dataset are incomplete and should not be interpreted as a real decline in business performance.

For future trend analysis, the most reliable period appears to be 2017-01 through 2018-08.

## Delivery Time Analysis

In [7]:
delivery_time_basics_query = """
WITH delivered_orders AS (
    SELECT
        o.order_id,
        julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS delivery_days
    FROM orders AS o
    WHERE o.order_status = 'delivered'
        AND o.order_delivered_customer_date IS NOT NULL
)
SELECT
    ROUND(AVG(delivery_days), 2) AS avg_delivery_days,
    ROUND(MIN(delivery_days), 2) AS min_delivery_days,
    ROUND(MAX(delivery_days), 2) AS max_delivery_days
FROM delivered_orders;
"""

delivery_time_basics_df = pd.read_sql_query(delivery_time_basics_query, conn)

delivery_time_basics_df

,avg_delivery_days,min_delivery_days,max_delivery_days
0,12.56,0.53,209.63


### Delivery Time Basics Observation

For delivered orders with valid delivery timestamps, the average delivery time is approximately 12.56 days. The minimum delivery time is less than one day, while the maximum delivery time is over 200 days.

The very high maximum suggests that the delivery-time distribution contains extreme outliers. Because of this, the average alone may not fully represent typical customer experience. Further analysis should inspect long delivery outliers and use additional metrics like delivery-time buckets or percentiles.

## Longest Delivery Outliers

In [8]:
longest_delivery_outliers_query = """
SELECT 
    o.order_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    ROUND(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp), 2) AS delivery_days
FROM orders AS o
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
ORDER BY delivery_days DESC
LIMIT 10;
"""

longest_delivery_outliers_df = pd.read_sql_query(
    longest_delivery_outliers_query,
    conn
)

longest_delivery_outliers_df

,order_id,order_purchase_timestamp,order_delivered_customer_date,delivery_days
0,ca07593549f1816d26a572e06dc1eab6,2017-02-21 23:31:27,2017-09-19 14:36:39,209.63
1,1b3190b2dfa9d789e1f14c05b647a14a,2018-02-23 14:57:35,2018-09-19 23:24:07,208.35
2,440d0d17af552815d15a9e41abe49359,2017-03-07 23:59:51,2017-09-19 15:12:50,195.63
3,2fb597c2f772eca01b1f5c561bf6cc7b,2017-03-08 18:09:02,2017-09-19 14:33:17,194.85
4,285ab9426d6982034523a855f55a885e,2017-03-08 22:47:40,2017-09-19 14:00:04,194.63
5,0f4519c5f1c541ddec9f21b3bddd533a,2017-03-09 13:26:57,2017-09-19 14:38:21,194.05
6,47b40429ed8cce3aee9199792275433f,2018-01-03 09:44:01,2018-07-13 20:51:31,191.46
7,2fe324febf907e3ea3f2aa9650869fa5,2017-03-13 20:17:10,2017-09-19 17:00:07,189.86
8,2d7561026d542c8dbd8f0daeadf67a43,2017-03-15 11:24:27,2017-09-19 14:38:18,188.13
9,c27815f7e3dd0b926b58552628481575,2017-03-15 23:23:17,2017-09-19 17:14:25,187.74


### Longest Delivery Outliers Observation

The longest delivered orders took around 188 to 210 days to reach the customer. This is not just a single outlier, since several orders have similarly extreme delivery durations.

Several of the longest 2017 cases were recorded as delivered around the same date, which may suggest a logistics issue, delayed status update, or data-entry pattern rather than normal delivery behavior.

Because of these extreme values, average delivery time should be interpreted carefully. Future analysis should use delivery-time buckets or percentiles to understand the typical customer experience.

## Delivery Time Buckets

In [9]:
delivery_time_buckets_query = """
WITH delivered_orders AS (
    SELECT 
        o.order_id,
        julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS delivery_days
    FROM orders AS o
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
),
bucketed_orders AS (
    SELECT
        order_id,
        delivery_days,
        CASE 
            WHEN delivery_days <= 7 THEN '0-7 days'
            WHEN delivery_days <= 14 THEN '8-14 days'
            WHEN delivery_days <= 30 THEN '15-30 days'
            WHEN delivery_days <= 60 THEN '31-60 days'
            ELSE '60+ days'
        END AS delivery_time_bucket,
        CASE 
            WHEN delivery_days <= 7 THEN 1
            WHEN delivery_days <= 14 THEN 2
            WHEN delivery_days <= 30 THEN 3
            WHEN delivery_days <= 60 THEN 4
            ELSE 5
        END AS bucket_sort_order
    FROM delivered_orders
)
SELECT 
    delivery_time_bucket,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS order_percentage
FROM bucketed_orders
GROUP BY delivery_time_bucket, bucket_sort_order
ORDER BY bucket_sort_order;
"""

delivery_time_buckets_df = pd.read_sql_query(delivery_time_buckets_query, conn)

delivery_time_buckets_df

,delivery_time_bucket,total_orders,order_percentage
0,0-7 days,26046,27.00
1,8-14 days,40212,41.68
2,15-30 days,25662,26.60
3,31-60 days,4244,4.40
4,60+ days,306,0.32


### Delivery Time Bucket Observation

Most delivered orders arrived within 14 days, with the largest group falling in the 8-14 day range. Around 95% of delivered orders arrived within 30 days.

Long deliveries exist, but they represent a small minority of orders. Only about 4.40% of delivered orders took 31-60 days, and only 0.32% took more than 60 days.

This means that extremely long delivery times are real, but they are not representative of the typical customer experience.


## Delivery Time and Review Scores

In [10]:
delivery_time_review_scores_query = """
WITH delivered_reviews AS (
    SELECT 
        o.order_id,
        r.review_id,
        r.review_score,
        julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS delivery_days
    FROM orders AS o
    INNER JOIN reviews AS r
        ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND r.review_score IS NOT NULL
),

bucketed_reviews AS (
    SELECT
        order_id,
        review_id,
        review_score,
        delivery_days,
        CASE 
            WHEN delivery_days <= 7 THEN '0-7 days'
            WHEN delivery_days <= 14 THEN '8-14 days'
            WHEN delivery_days <= 30 THEN '15-30 days'
            WHEN delivery_days <= 60 THEN '31-60 days'
            ELSE '60+ days'
        END AS delivery_time_bucket,
        CASE 
            WHEN delivery_days <= 7 THEN 1
            WHEN delivery_days <= 14 THEN 2
            WHEN delivery_days <= 30 THEN 3
            WHEN delivery_days <= 60 THEN 4
            ELSE 5
        END AS bucket_sort_order
    FROM delivered_reviews
)

SELECT 
    delivery_time_bucket,
    COUNT(*) AS total_reviews,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM bucketed_reviews
GROUP BY delivery_time_bucket, bucket_sort_order
ORDER BY bucket_sort_order;
"""
delivery_time_review_scores_df = pd.read_sql_query(delivery_time_review_scores_query, conn)

delivery_time_review_scores_df

,delivery_time_bucket,total_reviews,avg_review_score
0,0-7 days,26040,4.42
1,8-14 days,40214,4.31
2,15-30 days,25644,3.98
3,31-60 days,4163,2.26
4,60+ days,292,2.14


### Delivery Time and Review Score Observation

Average review scores decrease as delivery time increases. Orders delivered within 7 days have the highest average review score, while orders taking more than 30 days have much lower average scores.

The largest drop appears between the 15-30 day and 31-60 day delivery buckets, suggesting that customer satisfaction declines sharply when delivery takes longer than about one month.

This does not prove that delivery time alone causes lower reviews, but it shows a strong association between slower delivery and worse customer feedback.

## Late Delivery and Review Scores

In [11]:
late_delivery_reviews_query = """
WITH delivered_reviews AS (
    SELECT 
        o.order_id,
        r.review_id,
        r.review_score,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date
    FROM orders AS o
    INNER JOIN reviews AS r 
        ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
      AND r.review_score IS NOT NULL
),

status_mapped_reviews AS (
    SELECT 
        order_id,
        review_id,
        review_score,
        CASE 
            WHEN julianday(order_delivered_customer_date) <= julianday(order_estimated_delivery_date) THEN 'Early / On Time'
            ELSE 'Late'
        END AS delivery_status
    FROM delivered_reviews
)

SELECT 
    delivery_status,
    COUNT(*) AS total_reviews,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM status_mapped_reviews
GROUP BY delivery_status
ORDER BY avg_review_score DESC;
"""

late_delivery_reviews_df = pd.read_sql_query(late_delivery_reviews_query, conn)

late_delivery_reviews_df

,delivery_status,total_reviews,avg_review_score
0,Early / On Time,88653,4.29
1,Late,7700,2.57


### Late Delivery and Review Score Observation

Orders delivered early or on time received an average review score of 4.29, while late deliveries received an average review score of 2.57.

This is a major difference, suggesting that meeting the estimated delivery date is strongly associated with customer satisfaction. Late deliveries appear to be one of the clearest operational factors linked to lower review scores.

This does not prove that late delivery is the only cause of lower reviews, since product quality, seller behavior, and customer expectations may also affect ratings. However, delivery reliability is clearly an important part of the customer experience.

## Late Delivery Rate

In [12]:
late_delivery_rate_query = """
WITH status_mapped_orders AS (
    SELECT 
        o.order_id,
        CASE 
            WHEN julianday(o.order_delivered_customer_date) <= julianday(o.order_estimated_delivery_date) THEN 'Early / On Time'
            ELSE 'Late'
        END AS delivery_status
    FROM orders AS o
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
)

SELECT 
    delivery_status,
    COUNT(*) AS total_orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS order_percentage
FROM status_mapped_orders
GROUP BY delivery_status
ORDER BY total_orders DESC;
"""

late_delivery_rate_df = pd.read_sql_query(late_delivery_rate_query, conn)

late_delivery_rate_df

,delivery_status,total_orders,order_percentage
0,Early / On Time,88644,91.89
1,Late,7826,8.11


### Late Delivery Rate Observation

Most delivered orders arrived early or on time, representing 91.89% of delivered orders. Late deliveries account for 8.11% of delivered orders.

Although late deliveries are a minority, they are associated with much lower review scores based on the previous analysis. This suggests that delivery reliability may be a high-impact operational issue: it affects a relatively small share of orders, but those orders show significantly worse customer satisfaction.

## Monthly Late Delivery Rate

In [13]:
monthly_late_delivery_rate_query = """
SELECT 
    strftime('%Y-%m', o.order_purchase_timestamp) AS purchase_month,
    COUNT(*) AS total_delivered_orders,
    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
            ELSE 0 
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_order_percentage
FROM orders AS o
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY purchase_month
ORDER BY purchase_month ASC;
"""

monthly_late_delivery_rate_df = pd.read_sql_query(monthly_late_delivery_rate_query, conn)

monthly_late_delivery_rate_df

,purchase_month,total_delivered_orders,late_orders,late_order_percentage
0,2016-09,1,1,100.00
1,2016-10,265,3,1.13
2,2016-12,1,0,0.00
3,2017-01,750,23,3.07
4,2017-02,1653,53,3.21
5,2017-03,2546,142,5.58
6,2017-04,2303,181,7.86
7,2017-05,3545,128,3.61
8,2017-06,3135,121,3.86
9,2017-07,3872,133,3.43


### Monthly Late Delivery Rate Observation

Late delivery rates vary significantly across months. Most months have relatively low late delivery rates, but several months show major spikes, especially November 2017, February 2018, March 2018, and August 2018.

March 2018 has the highest meaningful late delivery rate, with 21.36% of delivered orders arriving after the estimated delivery date. This suggests that late delivery was not a constant issue, but instead became much worse during specific operational periods.

Very small months such as 2016-09 and 2016-12 should not be overinterpreted because they contain too few delivered orders.

## Late Delivery Rate by Customer State

In [14]:
late_delivery_rate_customer_state_query = """
SELECT 
    c.customer_state,
    COUNT(*) AS total_delivered_orders,
    SUM(
        CASE 
            WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1
            ELSE 0 
        END
    ) AS late_orders,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_order_percentage
FROM orders AS o
INNER JOIN customers AS c 
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY c.customer_state
HAVING COUNT(*) >= 500
ORDER BY late_order_percentage DESC;
"""

late_delivery_rate_customer_state_df = pd.read_sql_query(late_delivery_rate_customer_state_query, conn)

late_delivery_rate_customer_state_df

,customer_state,total_delivered_orders,late_orders,late_order_percentage
0,MA,717,141,19.67
1,CE,1279,196,15.32
2,BA,3256,457,14.04
3,RJ,12350,1664,13.47
4,PA,946,117,12.37
5,ES,1995,244,12.23
6,MS,701,81,11.55
7,PB,517,57,11.03
8,PE,1593,172,10.80
9,SC,3546,346,9.76


### Late Delivery Rate by Customer State Observation

Late delivery rates vary noticeably by customer state. Among states with at least 500 delivered orders, MA, CE, BA, RJ, and PA show some of the highest late delivery percentages.

This suggests that late delivery is not evenly distributed geographically. Some customer regions appear to experience worse delivery reliability than others.

However, this result should not be interpreted only as a distance problem yet. Since Olist is a marketplace with many sellers, late delivery may depend on seller location, customer location, logistics coverage, regional infrastructure, or overly optimistic estimated delivery dates. Further analysis using seller states or geolocation data would be needed to better explain the regional differences.

## Late Delivery Rate by Seller State

In [15]:
late_delivery_rate_by_state_query = """
SELECT 
    s.seller_state,
    COUNT(*) AS total_order_items,
    SUM(
        CASE 
            WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1
            ELSE 0 
        END
    ) AS late_order_items,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_item_percentage
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN sellers AS s
    ON oi.seller_id = s.seller_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY s.seller_state
HAVING COUNT(*) >= 500
ORDER BY late_item_percentage DESC;
"""

late_delivery_rate_by_state_df = pd.read_sql_query(late_delivery_rate_by_state_query, conn)

late_delivery_rate_by_state_df

,seller_state,total_order_items,late_order_items,late_item_percentage
0,SP,78598,6697,8.52
1,RJ,4685,380,8.11
2,DF,883,59,6.68
3,PR,8487,548,6.46
4,SC,4000,235,5.88
5,MG,8601,477,5.55
6,BA,624,34,5.45
7,RS,2169,97,4.47
8,GO,508,20,3.94


### Late Delivery Rate by Seller State Observation

Late delivery rates also vary by seller state. Among seller states with at least 500 delivered order items, SP has the highest late item percentage at 8.52%, followed by RJ, DF, and PR.

SP is especially important because it has by far the largest number of delivered order items while also having the highest late item percentage. This means SP is not only a high-volume seller state, but also a major contributor to late delivered items.

This analysis is item-level, not order-level, because joining orders to order items creates one row per item. Further analysis would be needed to determine whether lateness is driven by specific sellers, product categories, customer destinations, or seller-to-customer distance.

## Top Product Categories by Revenue

In [16]:
top_product_categories_by_revenue_query = """
SELECT 
    ct.product_category_name_english,
    ROUND(SUM(oi.price), 2) AS total_product_revenue,
    COUNT(*) AS total_items,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN products AS p 
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct 
    ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY ct.product_category_name_english
ORDER BY total_product_revenue DESC
LIMIT 10;
"""

top_product_categories_by_revenue_df = pd.read_sql_query(top_product_categories_by_revenue_query, conn)

top_product_categories_by_revenue_df

,product_category_name_english,total_product_revenue,total_items,avg_item_price
0,health_beauty,"1,233,131.72",9465,130.28
1,watches_gifts,"1,166,176.98",5859,199.04
2,bed_bath_table,"1,023,434.76",10953,93.44
3,sports_leisure,"954,852.55",8431,113.25
4,computers_accessories,"888,724.61",7644,116.26
5,furniture_decor,"711,927.69",8160,87.25
6,housewares,"615,628.69",6795,90.60
7,cool_stuff,"610,204.10",3718,164.12
8,auto,"578,966.65",4140,139.85
9,toys,"471,286.48",4030,116.94


### Top Product Categories by Revenue Observation

Health and beauty generated the highest product revenue among delivered orders, followed by watches/gifts and bed/bath/table. However, the top categories do not all follow the same revenue pattern.

Health and beauty combines high order-item volume with a relatively strong average item price, while watches/gifts generates high revenue with fewer items because of its higher average item price. Bed/bath/table appears more volume-driven, with the highest number of items among the top categories but a lower average item price.

This suggests that revenue performance is driven by both demand volume and item pricing, not just the number of items sold.

## Top Product Categories by Items Sold

In [17]:
top_product_categories_volume_query = """
SELECT 
    ct.product_category_name_english,
    COUNT(*) AS total_items,
    ROUND(SUM(oi.price), 2) AS total_product_revenue,
    ROUND(AVG(oi.price), 2) AS avg_item_price
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN products AS p 
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct 
    ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY ct.product_category_name_english
ORDER BY total_items DESC
LIMIT 10;
"""

top_product_categories_volume_df = pd.read_sql_query(top_product_categories_volume_query, conn)

top_product_categories_volume_df

,product_category_name_english,total_items,total_product_revenue,avg_item_price
0,bed_bath_table,10953,"1,023,434.76",93.44
1,health_beauty,9465,"1,233,131.72",130.28
2,sports_leisure,8431,"954,852.55",113.25
3,furniture_decor,8160,"711,927.69",87.25
4,computers_accessories,7644,"888,724.61",116.26
5,housewares,6795,"615,628.69",90.60
6,watches_gifts,5859,"1,166,176.98",199.04
7,telephony,4430,"309,860.23",69.95
8,garden_tools,4268,"470,495.28",110.24
9,auto,4140,"578,966.65",139.85


### Top Product Categories by Items Sold Observation

Bed/bath/table is the highest-volume category, with 10,953 delivered order items, followed by health/beauty and sports/leisure. This suggests that these categories are major demand drivers in the marketplace.

However, item volume alone does not explain revenue performance. Watches/gifts has fewer sold items than categories like bed/bath/table and health/beauty, but it generates very high product revenue because it has the highest average item price among the top-volume categories.

This shows that product category performance depends on both sales volume and pricing. Some categories are volume-driven, while others generate strong revenue through higher average item prices.

## Product Category Revenue Share

In [18]:
product_category_revenue_share_query = """
SELECT 
    ct.product_category_name_english,
    ROUND(SUM(oi.price), 2) AS total_product_revenue,
    ROUND(100.0 * SUM(oi.price) / SUM(SUM(oi.price)) OVER (), 2) AS revenue_percentage
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN products AS p 
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct 
    ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
GROUP BY ct.product_category_name_english
ORDER BY total_product_revenue DESC
LIMIT 10;
"""

product_category_revenue_share_df = pd.read_sql_query(product_category_revenue_share_query, conn)

product_category_revenue_share_df

,product_category_name_english,total_product_revenue,revenue_percentage
0,health_beauty,"1,233,131.72",9.45
1,watches_gifts,"1,166,176.98",8.94
2,bed_bath_table,"1,023,434.76",7.85
3,sports_leisure,"954,852.55",7.32
4,computers_accessories,"888,724.61",6.81
5,furniture_decor,"711,927.69",5.46
6,housewares,"615,628.69",4.72
7,cool_stuff,"610,204.10",4.68
8,auto,"578,966.65",4.44
9,toys,"471,286.48",3.61


### Product Category Revenue Share Observation

The top product categories contribute a large share of delivered product revenue, with the top 10 categories accounting for about 63% of total delivered product revenue.

Health/beauty is the largest category, but it represents only 9.45% of total product revenue. This suggests that revenue is not dominated by a single category, but instead spread across several strong categories such as watches/gifts, bed/bath/table, sports/leisure, and computers/accessories.

This indicates a relatively diversified revenue base, where both high-volume and higher-priced categories contribute meaningfully to overall product revenue.

## Product Category Review Performance

In [19]:
category_review_performance_query = """
SELECT
    ct.product_category_name_english,
    COUNT(*) AS total_item_reviews,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
INNER JOIN reviews AS r
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND r.review_score IS NOT NULL
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 500
ORDER BY avg_review_score ASC;
"""

category_review_performance_df = pd.read_sql_query(category_review_performance_query, conn)

category_review_performance_df

,product_category_name_english,total_item_reviews,avg_review_score
0,office_furniture,1664,3.52
1,bed_bath_table,10985,3.92
2,furniture_decor,8159,3.95
3,home_construction,593,3.96
4,computers_accessories,7672,3.98
5,telephony,4408,4.00
6,electronics,2711,4.07
7,watches_gifts,5825,4.07
8,baby,2967,4.08
9,garden_tools,4254,4.08


### Product Category Review Performance Observation

Average review scores vary across major product categories. Office furniture has the lowest average review score among categories with at least 500 item-review rows, followed by bed/bath/table.

Bed/bath/table is especially important because it appeared as one of the highest-volume and highest-revenue categories, but still has one of the lowest average review scores. This suggests that it may be a high-impact category for customer experience improvements.

In contrast, categories such as health/beauty and sports/leisure combine strong revenue or volume with relatively higher review scores. This shows that category performance should be evaluated using both business contribution and customer satisfaction, not revenue alone.

## Product Category Late Delivery

In [20]:
product_category_late_delivery_query = """
SELECT 
    ct.product_category_name_english,
    COUNT(*) AS total_items,
    SUM(
        CASE
            WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1
            ELSE 0
        END
    ) AS late_items,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_item_percentage
FROM orders AS o
INNER JOIN order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN products AS p
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 500
ORDER BY late_item_percentage DESC;
"""

product_category_late_delivery_df = pd.read_sql_query(product_category_late_delivery_query, conn)

product_category_late_delivery_df


,product_category_name_english,total_items,late_items,late_item_percentage
0,electronics,2729,266,9.75
1,health_beauty,9465,857,9.05
2,office_furniture,1668,149,8.93
3,baby,2982,262,8.79
4,musical_instruments,651,56,8.60
5,furniture_decor,8160,688,8.43
6,bed_bath_table,10953,920,8.40
7,telephony,4430,369,8.33
8,auto,4139,343,8.29
9,watches_gifts,5857,485,8.28


### Product Category Late Delivery Observation

Late delivery rates vary by product category. Electronics has the highest late item percentage among categories with at least 500 delivered items, followed by health/beauty, office furniture, baby, and musical instruments.

Health/beauty is especially important because it was also one of the highest-revenue and highest-volume categories. This suggests that even strong business categories can have operational weaknesses, and that improving delivery reliability in high-volume categories could affect many customers.

Bed/bath/table is also notable because it combines high sales volume, strong revenue contribution, relatively low review performance, and an above-average late item percentage. This makes it a category worth investigating further in the final analysis.

## Seller Performance Summary

In [21]:
seller_performance_query = """
SELECT 
    s.seller_id,
    s.seller_state,
    COUNT(*) AS total_item_reviews,
    ROUND(SUM(oi.price), 2) AS total_product_revenue,
    ROUND(
        100.0 * SUM(
            CASE
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_item_percentage,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN sellers AS s 
    ON oi.seller_id = s.seller_id
INNER JOIN reviews AS r 
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND r.review_score IS NOT NULL
GROUP BY s.seller_id, s.seller_state
HAVING COUNT(*) >= 100
ORDER BY total_product_revenue DESC
LIMIT 15;
"""

seller_performance_df = pd.read_sql_query(seller_performance_query, conn)

seller_performance_df

,seller_id,seller_state,total_item_reviews,total_product_revenue,late_item_percentage,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1140,"225,586.34",11.49,4.14
1,53243585a1d6dc2643021fd1853d8905,BA,398,"215,904.44",3.77,4.13
2,4a3ca9315b744ce9f8e9374361493884,SP,1949,"197,225.32",10.98,3.83
3,fa1c13f2614d7b5c4749cbc52fecda94,SP,575,"189,649.54",9.74,4.37
4,7c67e1448b00f6e969d365cea6b010ab,SP,1358,"186,664.01",9.50,3.35
5,7e93a43ef30c4f03f38b393420bc753a,SP,321,"165,751.50",5.30,4.36
6,da8622b14eb17ae2831f4ac5b9dab84a,SP,1565,"161,574.27",7.03,4.08
7,7a67c85e85bb2ce8582c35f2203ad736,SP,1151,"139,188.73",5.82,4.27
8,1025f0e2d44d7041d6cf58b6550e0bfa,SP,1422,"138,691.40",9.14,3.87
9,955fee9216a65b617aa5c0531780ce60,SP,1464,"130,823.82",7.99,4.09


### Seller Performance Summary Observation

The highest-revenue sellers are heavily concentrated in SP, suggesting that São Paulo is a major seller hub in this marketplace.

However, this query is at the item-review level because it joins `orders`, `order_items`, `sellers`, and `reviews`. For that reason, the volume metric is named `total_item_reviews` rather than `total_orders`.

This summary is useful for identifying high-revenue sellers that may also need monitoring for delivery reliability or customer satisfaction.

## Category Performance Summary

In [22]:
category_performance_summary_query = """
SELECT 
    ct.product_category_name_english,
    COUNT(*) AS total_item_reviews,
    ROUND(SUM(oi.price), 2) AS total_product_revenue,
    ROUND(AVG(oi.price), 2) AS avg_item_price,
    ROUND(
        100.0 * SUM(
            CASE 
                WHEN julianday(o.order_delivered_customer_date) > julianday(o.order_estimated_delivery_date) THEN 1 
                ELSE 0 
            END
        ) / COUNT(*),
        2
    ) AS late_item_percentage,
    ROUND(AVG(r.review_score), 2) AS avg_review_score
FROM orders AS o
INNER JOIN order_items AS oi 
    ON o.order_id = oi.order_id
INNER JOIN products AS p 
    ON oi.product_id = p.product_id
INNER JOIN category_translation AS ct 
    ON p.product_category_name = ct.product_category_name
INNER JOIN reviews AS r 
    ON o.order_id = r.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
  AND o.order_estimated_delivery_date IS NOT NULL
  AND r.review_score IS NOT NULL
GROUP BY ct.product_category_name_english
HAVING COUNT(*) >= 500
ORDER BY total_product_revenue DESC
LIMIT 15;
"""

category_performance_summary_df = pd.read_sql_query(
    category_performance_summary_query,
    conn
)

category_performance_summary_df

,product_category_name_english,total_item_reviews,total_product_revenue,avg_item_price,late_item_percentage,avg_review_score
0,health_beauty,9456,"1,228,986.81",129.97,8.86,4.19
1,watches_gifts,5823,"1,160,011.57",199.21,8.14,4.07
2,bed_bath_table,10985,"1,027,333.65",93.52,8.25,3.92
3,sports_leisure,8435,"954,117.99",113.11,7.30,4.17
4,computers_accessories,7671,"892,168.64",116.30,7.63,3.99
5,furniture_decor,8159,"712,414.53",87.32,8.27,3.95
6,housewares,6780,"613,843.72",90.54,6.36,4.11
7,cool_stuff,3698,"605,088.81",163.63,6.63,4.19
8,auto,4116,"572,632.84",139.12,8.24,4.12
9,garden_tools,4254,"468,477.18",110.13,7.90,4.08


### Category Performance Summary Observation

Health/beauty appears to be one of the strongest overall categories. It generates the highest product revenue among the top categories while maintaining a solid average review score of 4.19. However, it also has a relatively high late item percentage of 8.86%, suggesting that even strong revenue categories may still have delivery reliability issues.

Watches/gifts is also a major revenue contributor, but its performance is more price-driven. It has fewer item-review rows than categories such as bed/bath/table and health/beauty, but it has the highest average item price among the top categories at 199.21.

Bed/bath/table is one of the most important categories to monitor. It has the highest item-review volume and strong revenue contribution, but its average review score is only 3.92 and its late item percentage is 8.25%. This suggests that it is a high-impact category where customer experience improvements could matter.

Office furniture also stands out negatively. Although it has lower revenue than the leading categories, it has the lowest average review score in this summary at 3.52 and the highest late item percentage at 8.83%. This makes it a potential risk category from a customer satisfaction perspective.

Overall, category performance should not be judged by revenue alone. The strongest business categories are not always the strongest customer-experience categories, so revenue, item volume, delivery reliability, and review score should be evaluated together.

## SQL Analysis Summary

The SQL analysis shows that delivery reliability is strongly associated with customer satisfaction, and that business performance cannot be judged by revenue alone.

The strongest findings from this notebook are:

- Most orders are delivered, so completed-order analysis is meaningful.
- Order volume grew strongly through 2017 and remained high through most of 2018.
- Most delivered orders arrive within 30 days, but extreme delivery delays do exist.
- Late delivery is associated with much lower review scores.
- Late delivery problems vary by month and customer state.
- Some high-revenue product categories also show weaker customer-experience signals.
- Row grain matters: order-level, item-level, and item-review-level metrics must be interpreted differently.

These outputs are used in the visualization notebook to communicate the strongest insights more clearly.